# Post-hoc GNNExplainer on Vanilla GIN for MNIST

This notebook trains a vanilla GIN classifier on the sparsified MNIST dataset, then fits a post-hoc GNNExplainer and evaluates Jaccard@|GT| and Node AUROC.

In [1]:
import os
import sys
import random
import time
from pathlib import Path
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, f1_score
from tqdm import tqdm

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
print('Repo root added:', repo_root)

/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE = cpu
Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [2]:
def _augment_with_normalized_pos(dataset):
    augmented = []
    for data in dataset:
        d = data.clone()
        pos = d.pos.float()
        pos_min = pos.min(dim=0).values
        pos_max = pos.max(dim=0).values
        denom = (pos_max - pos_min).clamp(min=1e-8)
        pos_norm = (pos - pos_min) / denom
        d.x = torch.cat([d.x.float(), pos_norm], dim=-1)
        augmented.append(d)
    return augmented

def load_and_split_data(batch_size=128, add_pos_features=True):
    print("Loading preprocessed sparsified .pt splits...")
    candidate_roots = [
        repo_root / "data" / "MNIST" / "sparsified_pt_splits",
        repo_root / "shaique_updates" / "codes" / "MNISTsp" / "data" / "MNIST" / "sparsified_pt_splits",
    ]

    split_root = None
    for root in candidate_roots:
        if (root / "train_sparsified.pt").exists() and (root / "val_sparsified.pt").exists() and (root / "test_sparsified.pt").exists():
            split_root = root
            break
    if split_root is None:
        raise FileNotFoundError("Could not find saved sparsified split files.")

    train_dataset = torch.load(split_root / "train_sparsified.pt", map_location="cpu", weights_only=False)
    val_dataset = torch.load(split_root / "val_sparsified.pt", map_location="cpu", weights_only=False)
    test_dataset = torch.load(split_root / "test_sparsified.pt", map_location="cpu", weights_only=False)

    # Balance test_dataset to 1000 datapoints (100 per class)
    class_counts = {i: 0 for i in range(10)}
    balanced_test = []
    for data in test_dataset:
        label = int(data.y.item())
        if class_counts.get(label, 0) < 100:
            balanced_test.append(data)
            class_counts[label] = class_counts.get(label, 0) + 1
    test_dataset = balanced_test

    if add_pos_features:
        train_dataset = _augment_with_normalized_pos(train_dataset)
        val_dataset = _augment_with_normalized_pos(val_dataset)
        test_dataset = _augment_with_normalized_pos(test_dataset)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    all_labels = torch.tensor([int(d.y.item()) for d in train_dataset + val_dataset + test_dataset])
    num_classes = int(all_labels.unique().numel())
    num_node_features = int(train_dataset[0].num_node_features)

    dataset_info = {
        "num_node_features": num_node_features,
        "num_classes": num_classes,
    }

    print(f"Using split directory: {split_root}")
    print(f"Loaded splits. Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
    print(f"Node features: {num_node_features} (pos features added: {add_pos_features})")

    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader, dataset_info

BATCH_SIZE = 128
train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader, dataset_info = load_and_split_data(batch_size=BATCH_SIZE)

Loading preprocessed sparsified .pt splits...
Using split directory: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/shaique_updates/codes/MNISTsp/data/MNIST/sparsified_pt_splits
Loaded splits. Train: 20000, Val: 5000, Test: 1000
Node features: 3 (pos features added: True)


In [3]:
len(test_dataset)

1000

In [4]:
class VanillaGINBackbone(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=4):
        super().__init__()
        self.node_emb = nn.Linear(in_channels, hidden_channels)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_channels, 2 * hidden_channels),
                nn.BatchNorm1d(2 * hidden_channels),
                nn.ReLU(),
                nn.Linear(2 * hidden_channels, hidden_channels),
            )
            self.convs.append(GINConv(nn=mlp, train_eps=True))
            self.norms.append(nn.BatchNorm1d(hidden_channels))
        self.fc1 = nn.Linear(hidden_channels, hidden_channels // 2)
        self.fc2 = nn.Linear(hidden_channels // 2, out_channels)

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        _ = edge_attr
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        x = self.node_emb(x)
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = F.relu(norm(x))
        x = global_add_pool(x, batch)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.fc2(x)
        return x

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_graphs = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, edge_attr=getattr(batch, "edge_attr", None), batch=batch.batch)
        y = batch.y.view(-1).long()
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_graphs += batch.num_graphs
    return total_loss / max(total_graphs, 1), total_correct / max(total_graphs, 1)

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_graphs = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, edge_attr=getattr(batch, "edge_attr", None), batch=batch.batch)
        y = batch.y.view(-1).long()
        loss = criterion(logits, y)
        total_loss += loss.item() * batch.num_graphs
        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_graphs += batch.num_graphs
    return total_loss / max(total_graphs, 1), total_correct / max(total_graphs, 1)

def get_gnn_explainer(model, epochs: int = 200, lr: float = 0.01):
    return Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=epochs, lr=lr),
        explanation_type='model',
        node_mask_type='object',
        edge_mask_type=None,
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

In [5]:
def get_ground_truth_mask(data):
    keys_to_check = ['node_mask', 'explanation_mask']
    for key in keys_to_check:
        if hasattr(data, key):
            mask = getattr(data, key)
            if mask is not None:
                return mask
    return None

def evaluate_custom_jaccard(explainer, model, dataset, max_graphs=None):
    jaccard_scores = []
    graphs = dataset[:max_graphs] if max_graphs else dataset
    print(f"Evaluating Jaccard on {len(graphs)} graphs...")
    
    for data in tqdm(graphs):
        data = data.to(DEVICE)
        
        gt_mask = get_ground_truth_mask(data)
        if gt_mask is None: continue
        
        gt_mask = gt_mask.squeeze().cpu().numpy()
        gt_nodes = set(np.where(gt_mask == 1)[0])
        k = len(gt_nodes)
        if k == 0: continue

        with torch.no_grad():
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=DEVICE)
            logits = model(data.x, data.edge_index, edge_attr=getattr(data, "edge_attr", None), batch=batch)
            target = logits.argmax().item()
            
        explanation = explainer(
            x=data.x, 
            edge_index=data.edge_index, 
            target=torch.tensor([target], device=DEVICE),
            edge_attr=getattr(data, "edge_attr", None),
            batch=batch
        )
        
        pred_mask = explanation.node_mask.detach().cpu().numpy().flatten()
        
        top_k_indices = np.argsort(pred_mask)[-k:]
        pred_nodes = set(top_k_indices)

        intersection = len(gt_nodes.intersection(pred_nodes))
        union = len(gt_nodes.union(pred_nodes))
        jaccard_scores.append(intersection / (union + 1e-8))

    if not jaccard_scores: return 0.0
    return float(np.mean(jaccard_scores))

def evaluate_custom_auroc(explainer, model, dataset, max_graphs=None):
    auroc_scores = []
    graphs = dataset[:max_graphs] if max_graphs else dataset
    print(f"Evaluating AUROC on {len(graphs)} graphs...")
    
    for data in tqdm(graphs):
        data = data.to(DEVICE)
        
        gt_mask = get_ground_truth_mask(data)
        if gt_mask is None: continue
        gt_mask = gt_mask.squeeze().cpu().numpy()
        
        if gt_mask.sum() == 0 or gt_mask.sum() == len(gt_mask):
            continue

        with torch.no_grad():
            batch = torch.zeros(data.num_nodes, dtype=torch.long, device=DEVICE)
            logits = model(data.x, data.edge_index, edge_attr=getattr(data, "edge_attr", None), batch=batch)
            target = logits.argmax().item()

        explanation = explainer(
            x=data.x, 
            edge_index=data.edge_index, 
            target=torch.tensor([target], device=DEVICE),
            edge_attr=getattr(data, "edge_attr", None),
            batch=batch
        )
        
        pred_mask = explanation.node_mask.detach().cpu().numpy().flatten()
        
        try:
            score = roc_auc_score(gt_mask, pred_mask)
            auroc_scores.append(score)
        except ValueError: pass

    if not auroc_scores: return 0.0
    return float(np.mean(auroc_scores))

In [6]:
def _sync_if_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _get_batch(data):
    return data.batch if hasattr(data, "batch") else torch.zeros(data.num_nodes, dtype=torch.long, device=DEVICE)

def time_gnnexplainer_per_graph(gnn_explainer, model, dataset, warmup: int = 2, max_graphs: int | None = None):
    model.eval()
    times_ms = []
    graphs = dataset[:max_graphs] if max_graphs is not None else dataset

    # Warmup
    for data in graphs[:warmup]:
        data = data.to(DEVICE)
        batch = _get_batch(data)
        with torch.no_grad():
            logits = model(data.x, data.edge_index, edge_attr=getattr(data, "edge_attr", None), batch=batch)
            pred_label = logits.argmax(dim=-1).item()
        _ = gnn_explainer(x=data.x, edge_index=data.edge_index, edge_attr=getattr(data, "edge_attr", None), batch=batch, target=torch.tensor([pred_label], device=DEVICE))

    # Timing
    for data in graphs:
        data = data.to(DEVICE)
        batch = _get_batch(data)
        with torch.no_grad():
            logits = model(data.x, data.edge_index, edge_attr=getattr(data, "edge_attr", None), batch=batch)
            pred_label = logits.argmax(dim=-1).item()
        
        _sync_if_cuda()
        start = time.perf_counter()
        _ = gnn_explainer(x=data.x, edge_index=data.edge_index, edge_attr=getattr(data, "edge_attr", None), batch=batch, target=torch.tensor([pred_label], device=DEVICE))
        _sync_if_cuda()
        end = time.perf_counter()
        times_ms.append((end - start) * 1000)

    times_ms = np.asarray(times_ms, dtype=float)
    mean_ms = float(times_ms.mean()) if times_ms.size else float('nan')
    std_ms = float(times_ms.std()) if times_ms.size else float('nan')
    median_ms = float(np.median(times_ms)) if times_ms.size else float('nan')
    p90_ms = float(np.percentile(times_ms, 90)) if times_ms.size else float('nan')
    graphs_per_s = float(1000.0 / mean_ms) if mean_ms > 0 else float('nan')
    total_s = float(times_ms.sum() / 1000.0) if times_ms.size else float('nan')

    return {
        "mean_ms": mean_ms,
        "std_ms": std_ms,
        "median_ms": median_ms,
        "p90_ms": p90_ms,
        "graphs_per_s": graphs_per_s,
        "total_s": total_s,
        "n_graphs": int(times_ms.size),
    }

In [7]:
HIDDEN_DIM = 64
LR = 1e-3
EPOCHS = 100
EARLY_STOP_PATIENCE = 25

def run_multi_seed_gnnexplainer(seeds=(11, 22, 33, 44, 55)):
    results = []
    
    for seed in seeds:
        print(f"\n=== Seed {seed} ===")
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            
        model = VanillaGINBackbone(
            in_channels=dataset_info["num_node_features"],
            hidden_channels=HIDDEN_DIM,
            out_channels=dataset_info["num_classes"],
            num_layers=4,
        ).to(DEVICE)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=10, min_lr=1e-6
        )
        
        best_val_loss = float("inf")
        best_state = None
        epochs_no_improve = 0
        
        _sync_if_cuda()
        train_start = time.perf_counter()
        
        for epoch in range(1, EPOCHS + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
            val_loss, val_acc = eval_one_epoch(model, val_loader, criterion)
            scheduler.step(val_loss)
            
            if val_loss < best_val_loss - 1e-6:
                best_val_loss = val_loss
                best_state = deepcopy(model.state_dict())
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                
            if epoch % 5 == 0 or epoch == 1:
                print(f"  Epoch {epoch:03d}/{EPOCHS}: train_acc={train_acc:.4f} val_acc={val_acc:.4f} val_loss={val_loss:.4f}")
                
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break
                
        _sync_if_cuda()
        train_time_s = time.perf_counter() - train_start
        
        if best_state is not None:
            model.load_state_dict(best_state)
            
        print("Evaluating Explainer...")
        explainer = get_gnn_explainer(model, epochs=200, lr=0.01)
        
        test_loss, test_acc = eval_one_epoch(model, test_loader, criterion)
        
        # GNNExplainer is slow, limit to 1000 graphs or adjust as needed
        max_graphs_explainer = 1000
        node_jaccard = evaluate_custom_jaccard(explainer, model, test_dataset, max_graphs=max_graphs_explainer)
        node_auroc = evaluate_custom_auroc(explainer, model, test_dataset, max_graphs=max_graphs_explainer)
        timing = time_gnnexplainer_per_graph(explainer, model, test_dataset, warmup=2, max_graphs=max_graphs_explainer)
        
        print(f"Seed {seed} Results:")
        print(f"Test Acc: {test_acc:.4f} | Node Jaccard: {node_jaccard:.4f} | Node AUROC: {node_auroc:.4f}")
        print(f"Explainer Time: {timing['total_s']:.2f}s ({timing['mean_ms']:.2f} ms/graph)")
        
        results.append({
            "seed": seed,
            "test_acc": test_acc,
            "node_jaccard": node_jaccard,
            "node_auroc": node_auroc,
            "train_time_s": train_time_s,
            "explainer_total_s": timing["total_s"],
            "explainer_mean_ms": timing["mean_ms"]
        })
        
    return results

In [8]:
# Smoke test
#smoke_results = run_multi_seed_gnnexplainer(seeds=[123])
#print("Smoke test complete.")
#print(smoke_results)

SEEDS = [11, 22, 33]
results = run_multi_seed_gnnexplainer(seeds=SEEDS)
import pandas as pd
df = pd.DataFrame(results)
df



=== Seed 11 ===
  Epoch 001/100: train_acc=0.2931 val_acc=0.4176 val_loss=1.6682
  Epoch 005/100: train_acc=0.6693 val_acc=0.7204 val_loss=0.8544
  Epoch 010/100: train_acc=0.7765 val_acc=0.8364 val_loss=0.5100
  Epoch 015/100: train_acc=0.8102 val_acc=0.7858 val_loss=0.6662
  Epoch 020/100: train_acc=0.8277 val_acc=0.8758 val_loss=0.4077
  Epoch 025/100: train_acc=0.8358 val_acc=0.8556 val_loss=0.4870
  Epoch 030/100: train_acc=0.8485 val_acc=0.8640 val_loss=0.4458
  Epoch 035/100: train_acc=0.8547 val_acc=0.8600 val_loss=0.4488
  Epoch 040/100: train_acc=0.8622 val_acc=0.8848 val_loss=0.3804
  Epoch 045/100: train_acc=0.8653 val_acc=0.8914 val_loss=0.3592
  Epoch 050/100: train_acc=0.8679 val_acc=0.8994 val_loss=0.3456
  Epoch 055/100: train_acc=0.8918 val_acc=0.9124 val_loss=0.2975
  Epoch 060/100: train_acc=0.8952 val_acc=0.9108 val_loss=0.2953
  Epoch 065/100: train_acc=0.8986 val_acc=0.9064 val_loss=0.3052
  Epoch 070/100: train_acc=0.8980 val_acc=0.9074 val_loss=0.3235
  Epoch 

  0%|          | 0/1000 [00:00<?, ?it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:31: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
100%|██████████| 1000/1000 [05:54<00:00,  2.82it/s]


Evaluating AUROC on 1000 graphs...


  0%|          | 0/1000 [00:00<?, ?it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 1/1000 [00:00<09:11,  1.81it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 2/1000 [00:00<07:01,  2.37it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 3/1000 [00:01<06:25,  2.58it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 4/1000 [00:01<06:22,  2.60it/s]/var

Seed 11 Results:
Test Acc: 0.9040 | Node Jaccard: 0.3709 | Node AUROC: 0.6963
Explainer Time: 320.82s (320.82 ms/graph)

=== Seed 22 ===
  Epoch 001/100: train_acc=0.2653 val_acc=0.4150 val_loss=1.6968
  Epoch 005/100: train_acc=0.6728 val_acc=0.6700 val_loss=0.9983
  Epoch 010/100: train_acc=0.7751 val_acc=0.8148 val_loss=0.5706
  Epoch 015/100: train_acc=0.8048 val_acc=0.8442 val_loss=0.4833
  Epoch 020/100: train_acc=0.8226 val_acc=0.8196 val_loss=0.5546
  Epoch 025/100: train_acc=0.8366 val_acc=0.8524 val_loss=0.4783
  Epoch 030/100: train_acc=0.8430 val_acc=0.8668 val_loss=0.4305
  Epoch 035/100: train_acc=0.8450 val_acc=0.8778 val_loss=0.3878
  Epoch 040/100: train_acc=0.8578 val_acc=0.8766 val_loss=0.3980
  Epoch 045/100: train_acc=0.8790 val_acc=0.8942 val_loss=0.3435
  Epoch 050/100: train_acc=0.8841 val_acc=0.9008 val_loss=0.3330
  Epoch 055/100: train_acc=0.8821 val_acc=0.9024 val_loss=0.3258
  Epoch 060/100: train_acc=0.8902 val_acc=0.8944 val_loss=0.3509
  Epoch 065/100: t

...

  0%|          | 0/1000 [00:00<?, ?it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:31: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
100%|██████████| 1000/1000 [05:47<00:00,  2.88it/s]


Evaluating AUROC on 1000 graphs...


  0%|          | 0/1000 [00:00<?, ?it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 1/1000 [00:00<06:16,  2.65it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 2/1000 [00:00<05:44,  2.90it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 3/1000 [00:01<05:39,  2.94it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_3102/1499924367.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 4/1000 [00:01<05:34,  2.98it/s]/var

Seed 33 Results:
Test Acc: 0.9000 | Node Jaccard: 0.3049 | Node AUROC: 0.6038
Explainer Time: 333.84s (333.84 ms/graph)


,seed,test_acc,node_jaccard,node_auroc,train_time_s,explainer_total_s,explainer_mean_ms
0,11,0.904,0.370864,0.696338,629.315424,320.824270,320.824270
1,22,0.907,0.338319,0.635460,683.626788,326.423614,326.423614
2,33,0.900,0.304931,0.603776,690.042195,333.842861,333.842861


In [13]:
formatted_stats = df.apply(lambda col: f"{col.mean():.4f} ± {col.std():.4f}")

print(formatted_stats)

seed                  22.0000 ± 11.0000
test_acc                0.9037 ± 0.0035
node_jaccard            0.3380 ± 0.0330
node_auroc              0.6452 ± 0.0470
train_time_s         667.6615 ± 33.3632
explainer_total_s     327.0302 ± 6.5305
explainer_mean_ms     327.0302 ± 6.5305
dtype: object
